In [3]:
import sys
from pathlib import Path

sys.path.append(
    str(Path.cwd().parent / "src")
)

# -------------------------------
# BharatFlux
# -------------------------------

from preprocessing.loader import load_all
from preprocessing.cleaner import clean_all

# -------------------------------
# GEE Extraction
# -------------------------------

from extraction.gee import initialize
from extraction.sites import get_site
from extraction.products import get_product
from extraction.extractor import extract_timeseries

# -------------------------------
# Harmonization
# -------------------------------

from harmonization.temporal import (
    align_to_common_dates,
)

from harmonization.merge import (
    merge_observed_satellite,
)

# -------------------------------
# Benchmarking
# -------------------------------

from benchmarking.metrics import (
    calculate_metrics,
)

In [4]:
# Initialize Earth Engine
initialize()

# Data Directory
DATA_DIR = (
    Path.cwd().parent
    / "data"
    / "raw"
    / "bharatflux"
)

✓ Earth Engine initialized successfully.


In [5]:
# Load BharatFlux
datasets = load_all(DATA_DIR)

cleaned_datasets, validation_summary = clean_all(
    datasets
)

#Select Site
observed = cleaned_datasets[
    "BFT_2016_LE_ET_dmean"
].data

observed.head()

,DoY,LE,ET
0,1,121.533229,4.666875
1,2,124.119004,4.766169
2,3,104.577071,4.015759
3,4,109.743590,4.214153
4,5,110.055843,4.226144


In [6]:
# Extract MOD16
site = get_site("BFT")

product = get_product(
    "MOD16A2GF"
)

satellite = extract_timeseries(
    site,
    product,
    "2016-01-01",
    "2016-12-31",
)

satellite.head()

,Date,DoY,ET
0,2016-01-01,1,3.793601
1,2016-01-09,9,3.947705
2,2016-01-17,17,4.724370
3,2016-01-25,25,2.116354
4,2016-02-02,33,2.866710


In [7]:
# Temporal Alignment
observed_aligned, satellite_aligned = (
    align_to_common_dates(
        observed,
        satellite,
    )
)

# Merge
merged = merge_observed_satellite(
    observed_aligned,
    satellite_aligned,
)

merged.head()

,Date,DoY,Observed_LE,Observed_ET,Satellite_ET
0,2016-01-01,1,121.533229,4.666875,3.793601
1,2016-01-09,9,107.454253,4.126243,3.947705
2,2016-01-17,17,134.438082,5.162422,4.724370
3,2016-01-25,25,87.810117,3.371908,2.116354
4,2016-02-02,33,64.205003,2.465472,2.866710


In [8]:
# Benchmark Metrics
report = calculate_metrics(
    merged
)
report

MetricsReport(rmse=13.17177895120969, mae=8.924795656628286, bias=7.6163704169232735, correlation=0.7880338078657112, r2=0.6209972823393326)

In [9]:
# Pretty Print
print(f"RMSE        : {report.rmse:.3f}")
print(f"MAE         : {report.mae:.3f}")
print(f"Bias        : {report.bias:.3f}")
print(f"Correlation : {report.correlation:.3f}")
print(f"R²          : {report.r2:.3f}")

RMSE        : 13.172
MAE         : 8.925
Bias        : 7.616
Correlation : 0.788
R²          : 0.621
